In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
import numpy as np
import math
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any

# Import warnings
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from __future__ import annotations

# -- OPG-continuum: ride the overnight gap (momentum/continuation, the mirror of OPG-reversal's fade) -----------------------------------------------------------------------
#
# THE TRADE (operator's own spec, 2026-09-30/10-01, already implemented live in
# TradingBridgeApi/Services/Tape/Strategies/OPGReversal/TapeOPGReversalEngine.cs for the Reversal side of
# this family - this notebook's job is ONLY to publish the per-ticker statistics that engine's own
# OPGReversalGate.Check reads, not to re-implement live execution): "часове вікно для відправки ордерів
# 9:20-9:25. Вхід по ціні оупена. Розраховуємо піенел від значення гепа. Класи для рейтингів і виходу -
# 9:45, 10:00, 10:30."
#
# Concretely: read the stack's own Stack% during the 9:20-9:25 window - on this axis that IS the stock's
# overnight gap (LstPrcLstClsPct at the open print is, by construction, (open/prev_close - 1) * 100), no
# separate gap field needed. If |it| > 0.5, the gap is "strong" (positive) or "weak" (negative). Enter WITH the gap instead of against it: Long the strong (positive) one, Short the weak (negative) one, betting the gap keeps extending rather than closing - the exact opposite side of OPG-reversal's own trade on the identical 9:20-9:25 reading. Hold to a class-specific exit, all
# THREE on the SAME calendar morning: 9:45, 10:00, 10:30 - unlike CLO's own family (CLO-reversal.ipynb /
# CLO-continuum.ipynb), nothing here ever crosses midnight.
#
# WHY THIS MIRRORS CLO's OWN SKELETON SO CLOSELY: same final.parquet, same Stack% column, same nearest-
# checkpoint-read machinery, same alpha/gamma/lambda/sigma statistics - only the checkpoint clock and the
# pnl sign differ. OPG-reversal is this file's mirror image (fade the gap instead of riding it) - same checkpoints, same statistics, only the pnl sign in scan_day differs.
#
# SIGNAL WINDOW IS A RANGE, NOT AN ANCHOR+TOLERANCE (the one real shape difference from CLO's own
# checkpoints): the operator gave two clock ends (9:20..9:25), not a center-plus-tolerance - OPGReversalTiming
# .cs (the live config) keeps SignalFromMin/SignalToMin explicit for exactly this reason rather than
# reverse-engineering a center. The live engine's own ScanSignalWindowAsync scans this range FORWARD, minute
# by minute, and the ticker's SIGNAL reading is whichever minute's print is first seen - see `_first_in_window`
# below, the notebook's own copy of that same rule (gate-free here: the notebook is not deciding whether to
# trade, only recording what the first available print in the window was).
#
# ENTRY IS A SEPARATE ORDER TYPE (operator): "ми шлемо спеціальний ордер який заходить по ціні оупена" - it
# fills AT THE OPEN (9:30) regardless of which minute inside 9:20-9:25 it was sent from. The live engine
# reads a settled Gap% field for extra robustness against noisy early quotes (TapeOPGReversalEngine's own
# ResolveSettledGapPctBatchAsync); this notebook does not need that trick - final.parquet's own per-minute
# Stack% at/near 9:30 is already the clean historical record, so a plain nearest-9:30 read is the gap. No
# POISONED_CLOSE_SMIN-style exclude either: 9:30 is a real, intentionally-traded print here, not an artifact
# (confirmed against the live engine's own doc comment on BuildNearestSnapshotAsync).
#
# `dev_sig`/`sigma` (the ArbitRage/PairFlux normalisation) do not apply here, same as CLO: this file only
# ever reads the raw Stack% column and gates on a flat 0.5 pp trigger (OPGReversalGate's own published
# default for MinDevAbsShort/Long).
import math
from collections import namedtuple

import numpy as np
import pandas as pd

SESSION_SPLIT_MIN = 900   # 15:00 - inherited from CLO's own stage split; harmless here since every OPG-continuum
                          # checkpoint (9:20..10:30) sits well before it, so nothing ever gets treated as
                          # "late"/next-session the way CLO's own evening checkpoints do - see cn_cell_stage.py.

Checkpoint = namedtuple("Checkpoint", "name smin tol")

# SIGNAL is a RANGE (see header comment), not a Checkpoint - kept as its own pair of constants rather than
# forced into the anchor+tolerance shape ENTRY/EXIT use, matching OPGReversalTiming.Default's own
# SignalFromMin/SignalToMin split.
SIGNAL_FROM_MIN = 9 * 60 + 20   # 9:20
SIGNAL_TO_MIN = 9 * 60 + 25     # 9:25

# ENTRY/EXIT checkpoints: one shared SearchWindowMin (5), exactly OPGReversalTiming.Default's own value -
# live-matched, not CLO's per-checkpoint 5/10 split (that split existed in CLO to fit evening gaps many
# hours apart; OPG's own checkpoints are only 15-30 min apart, and the operator specified one window for
# all of them).
SEARCH_WINDOW_MIN = 5
_RAW_CHECKPOINTS = (
    ("ENTRY", (9, 30), SEARCH_WINDOW_MIN),
    ("EXIT0945", (9, 45), SEARCH_WINDOW_MIN),
    ("EXIT1000", (10, 0), SEARCH_WINDOW_MIN),
    ("EXIT1030", (10, 30), SEARCH_WINDOW_MIN),
)
EXIT_CLASSES = ("EXIT0945", "EXIT1000", "EXIT1030")

# Every exit lands the SAME calendar morning as ENTRY - no midnight crossing anywhere in this family (see
# OPGReversalTiming.Default's own doc comment: "no midnight wrap here at all"), so trade_date == sdate
# always, unlike CLO's own EXIT_DATE_OFFSET (-1 for the same-evening classes).
EXIT_DATE_OFFSET = {"EXIT0945": 0, "EXIT1000": 0, "EXIT1030": 0}


def _to_smin(hm, split=SESSION_SPLIT_MIN):
    t = hm[0] * 60 + hm[1]
    return t - 1440 if t >= split else t


def build_checkpoints(split=SESSION_SPLIT_MIN):
    """ENTRY + the three EXIT classes, each tolerance capped at half the gap to its nearest neighbour so
    two checkpoints can never claim the same print - same safety rule CLO's own build_checkpoints uses,
    just with uniform 5-min requests that the real 15/15/30-min gaps here never actually need to cap."""
    smins = [_to_smin(hm, split) for _, hm, _ in _RAW_CHECKPOINTS]
    out = {}
    for i, (name, hm, req) in enumerate(_RAW_CHECKPOINTS):
        prev_gap = (smins[i] - smins[i - 1]) if i > 0 else math.inf
        next_gap = (smins[i + 1] - smins[i]) if i + 1 < len(smins) else math.inf
        tol = min(req, prev_gap // 2, next_gap // 2)
        out[name] = Checkpoint(name, smins[i], int(tol))
    return out


def default_cfg(**over):
    cfg = dict(
        min_dev=0.5,          # the trigger: |9:20-9:25 reading| must exceed this - OPGReversalGate's own
                               # published default for MinDevAbsShort/Long
        gamma_floor=0.5, gamma_step=0.1, gamma_max=15.0, gamma_min_n=8, gamma_z=1.96,
        max_abs_pp=60.0,      # a print beyond this is a broken quote, not a price
    )
    cfg.update(over)
    return cfg


def panel_from_stack(tickers, t_first, stack, prev_close=None):
    """One value per ticker per minute - Stack% itself (+ optionally the raw `prev_close` it was measured
    against, same shape, for REBASE_TO_TODAYS_CLOSE below). `t_first` is the SMIN of column 0."""
    out = dict(tickers=np.asarray(tickers), t_first=int(t_first), stack=np.asarray(stack, np.float32))
    if prev_close is not None:
        out["prev_close"] = np.asarray(prev_close, np.float32)
    return out


def _clean(stack, cfg):
    bad = ~np.isfinite(stack) | (np.abs(stack) > cfg["max_abs_pp"])
    out = stack.copy()
    out[bad] = np.nan
    return out


def _first_in_window(mat, t_first, lo_min, hi_min):
    """SIGNAL's own read: scan lo_min..hi_min FORWARD, minute by minute, and take the first finite print
    per ticker - the notebook's copy of TapeOPGReversalEngine.ScanSignalWindowAsync's own "forward-only,
    first qualifying minute wins" rule (gate-free here: every ticker with ANY print in the window gets a
    reading, the min_dev/gamma gate is applied afterward in scan_day, same as CLO's own `have` mask).
    Sentinel minute -1 (not "no match", since real smin values here are all positive on this axis - see
    `_read_at`-style callers elsewhere for why -1 is checked as an exact value)."""
    N, T = mat.shape
    val = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    lo_j = max(0, lo_min - t_first)
    hi_j = min(T - 1, hi_min - t_first)
    for j in range(lo_j, hi_j + 1):
        if remaining.size == 0:
            break
        v = mat[remaining, j]
        hit = np.isfinite(v)
        if hit.any():
            idx = remaining[hit]
            val[idx] = v[hit]
            used[idx] = j
            remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return val, minute


def _nearest(mat, t_first, t_col, tol, exclude=None, prefer_later=False):
    """value + the absolute minute actually used, per row, searching outward from t_col: 0, -1, +1, -2, +2,
    ... (or 0, +1, -1, +2, -2, ... when `prefer_later=True`) so a tie between an earlier and a later print
    keeps the preferred side - the first ring that finds a print wins, and a row already filled is never
    looked at again.

    `prefer_later`: ENTRY uses False (the live engine's own BuildNearestSnapshotAsync calls it with
    preferLaterOnTie=false for ENTRY), EXIT classes use True (preferLaterOnTie=true there) - see
    TapeOPGReversalEngine.cs's own two call sites for exactly this split; CLO's own `_nearest` never needed
    the parameter because it only ever called it one way.

    `exclude`: raw minute indices that are never accepted as a match even if finite. Unused by this file
    (no POISONED_CLOSE_SMIN-equivalent here - kept for shape parity with CLO's own `_nearest`).

    Sentinel: a row with NO match anywhere in the window gets `minute == -1` (the literal integer, NOT
    `-1 + t_first`).
    """
    N, T = mat.shape
    val = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    j0 = t_col - t_first
    excl_j = {m - t_first for m in exclude} if exclude else set()
    for d in range(0, int(tol) + 1):
        if remaining.size == 0:
            break
        order = (j0,) if d == 0 else ((j0 + d, j0 - d) if prefer_later else (j0 - d, j0 + d))
        for j in order:
            if remaining.size == 0:
                break
            if j < 0 or j >= T or j in excl_j:
                continue
            v = mat[remaining, j]
            hit = np.isfinite(v)
            if hit.any():
                idx = remaining[hit]
                val[idx] = v[hit]
                used[idx] = j
                remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return val, minute


def scan_day(panel, cfg, checkpoints):
    """-> (trades, signals, entries) for one session (one calendar morning - no midnight crossing in this
    family, see EXIT_DATE_OFFSET above).
    trades: one row per ticker per exit class that had BOTH a triggering signal (|dev|>min_dev) and a valid
    entry AND a valid exit for that class.
    signals: the RAW 9:20-9:25 reading for every ticker that had one, gated on nothing - ALPHA's source.
    entries: the RAW ENTRY-checkpoint (9:30) Stack% reading for every ticker that had one, gated on
    nothing - LAMBDA's source. Unlike CLO, there is no REBASE_TO_TODAYS_CLOSE step here: ENTRY and every
    EXIT sit on the SAME calendar morning, measured against the SAME already-settled prev_close (the
    rollover that matters already happened hours earlier), so entries == the traded entry_val directly -
    nothing to rebase."""
    stack = _clean(panel["stack"], cfg)
    t_first = panel["t_first"]
    tk = panel["tickers"]

    sig_v, sig_m = _first_in_window(stack, t_first, SIGNAL_FROM_MIN, SIGNAL_TO_MIN)
    ent_v, ent_m = _nearest(stack, t_first, checkpoints["ENTRY"].smin, checkpoints["ENTRY"].tol,
                             prefer_later=False)

    # LAMBDA's population: the ENTRY print itself - no pre/post-rebase distinction to make here (see
    # docstring above), so `entries` and the trades' own `entry_val` are the SAME number, unlike CLO.
    ent_ok = np.isfinite(ent_v)
    entries = pd.DataFrame({"ticker": tk[ent_ok], "entry_raw": ent_v[ent_ok]})

    sig_ok = np.isfinite(sig_v)
    signals = pd.DataFrame({"ticker": tk[sig_ok], "dev": sig_v[sig_ok]})

    have = sig_ok & np.isfinite(ent_v) & (np.abs(sig_v) > cfg["min_dev"])
    if not have.any():
        return _EMPTY_TRADES.copy(), signals, entries

    idx0 = np.flatnonzero(have)

    rows = []
    for cls in EXIT_CLASSES:
        ck = checkpoints[cls]
        ex_v, ex_m = _nearest(stack, t_first, ck.smin, ck.tol, prefer_later=True)
        ex_ok = np.isfinite(ex_v[idx0])
        if not ex_ok.any():
            continue
        j = idx0[ex_ok]
        e_dev = sig_v[j]
        e_val = ent_v[j]
        x_val = ex_v[j]
                # CONTINUUM's own thesis, the mirror of OPG-reversal: Long on a positive (strong) gap, Short on a negative (weak) one - pnl = +sign(dev)*(exit-entry), the exact negative of OPG-reversal's own formula.
        pnl = np.sign(e_dev) * (x_val - e_val)
        rows.append(pd.DataFrame({
            "ticker": tk[j], "cls": cls, "sign": np.where(e_dev > 0, "pos", "neg"),
            "entry_dev": e_dev, "entry_val": e_val, "exit_val": x_val, "pnl": pnl.astype(np.float32),
            "signal_minute": sig_m[j], "entry_minute": ent_m[j], "exit_minute": ex_v[j] * 0 + ex_m[j],
            "date_offset": EXIT_DATE_OFFSET[cls],
        }))
    trades = pd.concat(rows, ignore_index=True) if rows else _EMPTY_TRADES.copy()
    return trades, signals, entries


_EMPTY_TRADES = pd.DataFrame(columns=["ticker", "cls", "sign", "entry_dev", "entry_val", "exit_val", "pnl",
                                       "signal_minute", "entry_minute", "exit_minute", "date_offset"])


def _add_days(sdate_int, offset_days):
    """int YYYYMMDD + a signed day offset -> int YYYYMMDD. Always a no-op in this family
    (EXIT_DATE_OFFSET is all zeros) - kept for shape parity with CLO's own scan_days."""
    if offset_days == 0:
        return sdate_int
    ts = pd.Timestamp(str(sdate_int)) + pd.Timedelta(days=int(offset_days))
    return int(ts.strftime("%Y%m%d"))


def scan_days(day_iter, cfg, checkpoints=None):
    checkpoints = checkpoints or build_checkpoints()
    tr_all, sg_all, en_all = [], [], []
    for sdate, panel in day_iter:
        tr, sg, en = scan_day(panel, cfg, checkpoints)
        if len(tr):
            tr = tr.copy()
            tr.insert(0, "sdate", int(sdate))
            tr["trade_date"] = [_add_days(int(sdate), off) for off in tr["date_offset"]]
            tr_all.append(tr.drop(columns="date_offset"))
        if len(sg):
            sg = sg.copy()
            sg.insert(0, "sdate", int(sdate))
            sg_all.append(sg)
        if len(en):
            en = en.copy()
            en.insert(0, "sdate", int(sdate))
            en_all.append(en)
    trades = pd.concat(tr_all, ignore_index=True) if tr_all else (
        _EMPTY_TRADES.copy().assign(sdate=pd.Series(dtype=int), trade_date=pd.Series(dtype=int))
        [["sdate", *[c for c in _EMPTY_TRADES.columns if c != "date_offset"], "trade_date"]])
    signals = pd.concat(sg_all, ignore_index=True) if sg_all else pd.DataFrame(columns=["sdate", "ticker", "dev"])
    entries = pd.concat(en_all, ignore_index=True) if en_all else pd.DataFrame(columns=["sdate", "ticker", "entry_raw"])
    return trades, signals, entries


In [ ]:
from __future__ import annotations

# -- ratings, alpha, gamma, lambda from the trade/signal/entries tables scan_days produces -----------------
# Identical machinery to CLO's own rv_cell_ratings.py/cn_cell_ratings.py - alpha/gamma/lambda are all
# computed purely from the trades/signals/entries tables, which carry no notion of fade vs ride themselves;
# the only place OPG-continuum's own thesis differs from OPG-reversal's is the `pnl` sign in this file's
# sibling core cell's scan_day.
import math

import numpy as np
import pandas as pd


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _modal_value(values, step=0.1, min_n=3):
    """The value a stack gives MOST OFTEN, i.e. the mode of the reading, not its RMS. |values| are binned at
    the same 0.1pp granularity GAMMA already walks in (gamma_step), the modal bin wins, and the reported
    value is the MEAN of the raw readings that landed in it. NaN below min_n readings."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < min_n:
        return float("nan")
    bins = np.round(v / step) * step
    uniq, counts = np.unique(bins, return_counts=True)
    best = uniq[np.argmax(counts)]
    in_bin = np.abs(bins - best) < 1e-9
    return float(np.mean(v[in_bin]))


def compute_alpha(signals, step=0.1, min_n=3):
    """ALPHA, per ticker, split pos/neg: the modal |9:20-9:25 reading| on each side, over EVERY reading the
    ticker produced - whether or not it was big enough to trade, same gate-free universe CLO uses.
    -> DataFrame indexed by ticker, columns alpha_pos/alpha_neg."""
    if not len(signals):
        return pd.DataFrame(columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")
    out = {}
    for tk, g in signals.groupby("ticker"):
        dev = g["dev"].to_numpy()
        out[tk] = (
            _modal_value(dev[dev > 0], step, min_n),
            _modal_value(-dev[dev < 0], step, min_n),
        )
    return pd.DataFrame.from_dict(out, orient="index", columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")


def compute_lambda(entries, min_n=2):
    """LAMBDA (λ), per ticker: sample std (ddof=1) of the RAW ENTRY-checkpoint Stack% reading (9:30 - the
    open print, i.e. the gap itself). Unlike CLO, OPG-continuum's own `entries` table already IS the
    traded entry_val (no pre/post-rebase split - see scan_day's own docstring), so this is literally the
    std of the ticker's own historical open-gaps. Gate-free, same universe as ALPHA. NaN below min_n
    readings. -> Series indexed by ticker, named "lambda"."""
    if not len(entries):
        return pd.Series(dtype=float, name="lambda").rename_axis("ticker")
    counts = entries.groupby("ticker")["entry_raw"].transform("count")
    out = entries[counts >= min_n].groupby("ticker")["entry_raw"].std(ddof=1)
    return out.rename("lambda").rename_axis("ticker")


def compute_ratings(trades):
    """total/wins/win_rate(+Wilson LB)/mean pnl(+95% LB)/median, per (ticker, cls, sign)."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "total", "wins", "win_rate", "win_rate_lb",
                                     "mean_pnl", "median_pnl", "pnl_lb"])
    g = trades.groupby(["ticker", "cls", "sign"])
    out = g["pnl"].agg(total="size", mean_pnl="mean", median_pnl="median", sd_pnl="std").reset_index()
    out["wins"] = g["pnl"].apply(lambda s: int((s > 0).sum())).to_numpy()
    out["win_rate"] = out["wins"] / out["total"]
    out["win_rate_lb"] = _wilson_lb(out["wins"], out["total"])
    with np.errstate(all="ignore"):
        se = out["sd_pnl"] / np.sqrt(out["total"])
        out["pnl_lb"] = np.where(out["total"] > 1, out["mean_pnl"] - 1.96 * se, np.nan)
    return out.drop(columns="sd_pnl")


def compute_gamma(trades, cfg):
    """GAMMA, per (ticker, cls, sign): the LOWEST level L >= gamma_floor at which trades reaching |entry_dev|
    >= L have a positive 95% lower bound on mean pnl - OPGReversalGate.Check's own "the lowest deviation
    level at which this ticker has actually reversed profitably" (its doc comment's wording; OPG-continuum
    reads `pnl`, so its own sign convention flows through automatically, same as CLO)."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"])
    rows = []
    for (tk, cls, sg), g in trades.groupby(["ticker", "cls", "sign"]):
        dev = np.abs(g["entry_dev"].to_numpy())
        pnl = g["pnl"].to_numpy()
        lvl = float(cfg["gamma_floor"])
        found = None
        while lvl <= cfg["gamma_max"] + 1e-9:
            m = dev >= lvl - 1e-9
            n = int(m.sum())
            if n < cfg["gamma_min_n"]:
                break
            mp = float(pnl[m].mean())
            se = float(pnl[m].std(ddof=1)) / math.sqrt(n) if n > 1 else math.inf
            lb = mp - cfg["gamma_z"] * se
            if lb > 0:
                found = (lvl, n, mp, float((pnl[m] > 0).mean()))
                break
            lvl = round(lvl + cfg["gamma_step"], 6)
        if found is not None:
            rows.append({"ticker": tk, "cls": cls, "sign": sg, "gamma": found[0], "gamma_n": found[1],
                        "gamma_mean": found[2], "gamma_rate": found[3]})
    return pd.DataFrame(rows, columns=["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"])


def build_summary(trades, signals, cfg, sigma_by_ticker=None, lambda_by_ticker=None):
    """alpha + ratings + gamma, joined into one per-(ticker, cls, sign) table. "pos"/"neg" label the
    DIRECTION OF THE 9:20-9:25 EXTREME (the gap's own sign), not the trade taken - identical labelling to
    CLO even though a "pos" row here is a LONG (ride the strong/positive gap). `sigma` is read from the
    same final.parquet column every strategy reads (ticker property, not strategy-specific). `lambda` is
    `compute_lambda`'s own per-ticker Series, computed fresh each run from this run's own entries table."""
    alpha = compute_alpha(signals)
    ratings = compute_ratings(trades)
    gamma = compute_gamma(trades, cfg)
    out = ratings.merge(gamma, on=["ticker", "cls", "sign"], how="left")

    alpha_pos = out["ticker"].map(alpha["alpha_pos"]) if len(alpha) else np.nan
    alpha_neg = out["ticker"].map(alpha["alpha_neg"]) if len(alpha) else np.nan
    out["alpha"] = np.where(out["sign"] == "pos", alpha_pos, alpha_neg)

    if sigma_by_ticker:
        out["sigma"] = out["ticker"].map(sigma_by_ticker)
    else:
        out["sigma"] = np.nan

    if lambda_by_ticker is not None and len(lambda_by_ticker):
        out["lambda"] = out["ticker"].map(lambda_by_ticker)
    else:
        out["lambda"] = np.nan

    return out, alpha


In [ ]:
from __future__ import annotations

# -- stage: final.parquet -> one small parquet per SESSION ---------------------------------------------------
# Same staging strategy CLO uses - OPG-continuum reads the exact same final.parquet, the exact same
# Stack%/sigma columns. Renamed (opgcontinuum_ instead of rv_/cn_) so every strategy's own stage
# directory/sidecar lives on disk without colliding with any other.
#
# prev_close is NOT staged here (unlike CLO): OPG-continuum never rebases anything (see scan_day's own
# docstring in the core cell - ENTRY and every EXIT share the same already-settled reference close), so
# there is nothing for a staged prev_close column to be used for.
import numpy as np
import pandas as pd

# SESSION_SPLIT_MIN and panel_from_stack come from this notebook's own core cell, executed into the same
# namespace before this cell (the notebook order) - not imported, same pattern CLO's own stage cell uses.


def opgcontinuum_stage_by_session(input_path, stage_dir, *, session_split_min=SESSION_SPLIT_MIN, start_date=None,
                        STOCK_NUM_FIELD="Stack%", SIGMA_FIELD="sigma",
                        smin_lo=520, smin_hi=650, log_every_n_chunks=20):
    """-> (sorted session dates, {ticker: static sigma}).

    Sigma is READ, not computed: the source final.parquet already carries a per-row "sigma" column, the
    same static-per-ticker dispersion figure every strategy reads. The LATEST non-null value seen per
    ticker wins (row groups are read in file order, which is time order).

    smin_lo/smin_hi default to 520/650 (8:40..10:50) - tight around this family's own 9:20-10:30 window,
    unlike CLO's -520/650 (which has to reach back to the prior evening's 15:50 signal too)."""
    import gc, time, shutil
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    start_i = int(start_date.replace("-", "")) if start_date else -1

    schema = pa.schema([("ticker", pa.string()), ("smin", pa.int16()), ("stack", pa.float32())])
    writers, counts = {}, {}
    sigma_by_ticker: dict = {}

    def _writer(sdate):
        if sdate not in writers:
            writers[sdate] = pq.ParquetWriter(str(stage / f"{sdate}.parquet"), schema, compression="zstd")
            counts[sdate] = 0
        return writers[sdate]

    pf = pq.ParquetFile(input_path)
    want = ["ticker", "dt", STOCK_NUM_FIELD]
    cols = [c for c in want if c in pf.schema.names]
    for need in want:
        if need not in cols:
            raise KeyError(f"final.parquet is missing required column: {need}")
    has_sigma = SIGMA_FIELD in pf.schema.names
    if has_sigma:
        cols = cols + [SIGMA_FIELD]
    else:
        print(f"WARN: final.parquet has no '{SIGMA_FIELD}' column - sigma will be null for every ticker")
    t0 = time.time()
    total_in = total_out = 0
    print(f"START opgcontinuum stage  file={input_path}  session_split={session_split_min}min")
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)
            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt, df = dt[ok], df.loc[ok]
            t_arr = dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 + dt.dt.minute.to_numpy(dtype="int32", copy=False)
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 1440, t_arr).astype("int16")
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 + sess.dt.month.to_numpy(dtype="int32", copy=False) * 100
                     + sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")
            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            if has_sigma:
                sigma_col = pd.to_numeric(df[SIGMA_FIELD], errors="coerce").to_numpy(dtype="float64", copy=False)
                tick_col = df["ticker"].to_numpy(copy=False)
                for tk, sv in zip(tick_col, sigma_col):
                    if np.isfinite(sv):
                        sigma_by_ticker[str(tk)] = float(sv)
            keep = (smin >= smin_lo) & (smin <= smin_hi) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if not keep.any():
                continue
            out = pd.DataFrame({"ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str), "sdate": sdate[keep],
                                "smin": smin[keep], "stack": stack[keep]})
            for sd, part in out.groupby("sdate", sort=False):
                _writer(int(sd)).write_table(pa.Table.from_pandas(part[["ticker", "smin", "stack"]], schema=schema, preserve_index=False))
                counts[int(sd)] += len(part)
                total_out += len(part)
            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                print(f"[rg {ci + 1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} sessions={len(writers)} elapsed={time.time() - t0:.1f}s")
                gc.collect()
    finally:
        for w in writers.values():
            w.close()
    print(f"DONE stage in={total_in:,} staged={total_out:,} sessions={len(writers)} "
          f"sigma_tickers={len(sigma_by_ticker):,} elapsed={time.time() - t0:.1f}s")
    return sorted(counts), sigma_by_ticker


def opgcontinuum_sessions(stage_dir, *, smin_lo=520, smin_hi=650, min_tickers=30):
    """yields (sdate, panel) in date order, one pivoted session at a time."""
    from pathlib import Path
    for f in sorted(Path(stage_dir).glob("*.parquet"), key=lambda p: int(p.stem)):
        df = pd.read_parquet(f)
        if df["ticker"].nunique() < min_tickers:
            continue
        df = df.drop_duplicates(["ticker", "smin"], keep="last")
        tick, ti = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        T = smin_hi - smin_lo + 1
        stack = np.full((tick.size, T), np.nan, np.float32)
        stack[ti, df["smin"].to_numpy().astype(np.int64) - smin_lo] = df["stack"].to_numpy()
        yield int(f.stem), panel_from_stack(tick, smin_lo, stack)


In [ ]:
from pathlib import Path
import os, gzip, json, datetime


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


# -- runner ---------------------------------------------------------------------------------------------------
# Strategy code "opgcontinuum" - matches TradingBridgeApi.Services.Strategy.OPGReversal.OPGReversalRatingsService's
# own `private const string Strategy = "opgreversal"` / RelPath "opgreversal/opgreversal_rolling_perf.json.gz"
# EXACTLY (that service is already built and waiting for this exact file - see its own doc comment, "no
# loader exists yet ... until the operator's own notebook publishes an opgreversal_rolling_perf.json.gz-
# shaped file"). No OPGContinuum backend exists yet (checked: no OPGContinuum*/opgcontinuum hits anywhere in AXION-app) - "opgcontinuum" is this notebook's own choice, following the exact no-separator-lowercase convention OPGReversalRatingsService.cs already established for "opgreversal", so a future backend port can reuse it verbatim.
FINAL_PATH, OUT_DIR = _resolve_orion_paths("opgcontinuum")

STAGE_DIR = OUT_DIR.parent.parent / "_work" / "opgcontinuum_stage"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
SIGMA_SIDECAR = STAGE_DIR.parent / "opgcontinuum_stage_sigma.json"

_stage_ok = STAGE_DIR.exists() and any(STAGE_DIR.glob("*.parquet")) and SIGMA_SIDECAR.exists() and \
    min(p.stat().st_mtime for p in STAGE_DIR.glob("*.parquet")) >= FINAL_PATH.stat().st_mtime
if _stage_ok:
    print("stage is at least as new as final.parquet (incl. its sigma sidecar) - reusing", STAGE_DIR)
    sigma_by_ticker = json.loads(SIGMA_SIDECAR.read_text(encoding="utf-8"))
else:
    _, sigma_by_ticker = opgcontinuum_stage_by_session(str(FINAL_PATH), str(STAGE_DIR))
    SIGMA_SIDECAR.write_text(json.dumps(sigma_by_ticker), encoding="utf-8")
    if not sigma_by_ticker:
        print("WARN: fresh stage still produced no sigma for any ticker - check final.parquet's own "
              "'sigma' column")

CFG = default_cfg()
CHECKPOINTS = build_checkpoints()

trades, signals, entries = scan_days(opgcontinuum_sessions(str(STAGE_DIR)), CFG, CHECKPOINTS)
print(f"trades: {len(trades):,} over {trades['sdate'].nunique() if len(trades) else 0} sessions, "
      f"{trades['ticker'].nunique() if len(trades) else 0} tickers")

lambda_by_ticker = compute_lambda(entries)
print(f"lambda: {len(lambda_by_ticker):,} tickers")

summary, alpha = build_summary(trades, signals, CFG, sigma_by_ticker, lambda_by_ticker)
summary.to_csv(OUT_DIR / "opgcontinuum_ratings.csv", index=False)

# the scout file - same shape every other strategy's own rolling_perf.json.gz uses (ReversalRollingPerfParser
# is the byte-for-byte reference; ContinuumRollingPerfParser is a verified port of it - a future
# OPGReversalRollingPerfParser.cs should be a straight port too, same "cls" UPPERCASE / "sign" pos/neg wire
# convention this file writes below).
trade_dates_sorted = sorted(trades["trade_date"].unique().tolist()) if len(trades) else []
recent_dates = [str(s) for s in trade_dates_sorted][-255:]
date_index = {int(d): i for i, d in enumerate(recent_dates)}
WINDOWS_DAYS = [5, 20, 40, 65]

CLASS_ORDER = ["EXIT0945", "EXIT1000", "EXIT1030"]


def _rate(r):
    return None if pd.isna(r) else round(float(r), 4)


def _lambda_of(tk):
    return None if tk not in lambda_by_ticker.index else round(float(lambda_by_ticker[tk]), 4)


ROLLING_PATH = OUT_DIR / "opgcontinuum_rolling_perf.json.gz"
by_ticker_ratings: dict = {}
for _, r in summary.iterrows():
    by_ticker_ratings.setdefault(r["ticker"], []).append({
        "cls": r["cls"], "sign": r["sign"], "total": int(r["total"]), "wins": int(r["wins"]),
        "win_rate": _rate(r["win_rate"]), "win_rate_lb": _rate(r["win_rate_lb"]),
        "mean_pnl": _rate(r["mean_pnl"]), "pnl_lb": _rate(r["pnl_lb"]),
        "gamma": _rate(r["gamma"]), "gamma_n": None if pd.isna(r["gamma_n"]) else int(r["gamma_n"]),
        "gamma_rate": _rate(r["gamma_rate"]),
    })


def _alpha_side(tk, col):
    if tk not in alpha.index:
        return None
    v = alpha.loc[tk, col]
    return None if pd.isna(v) else round(float(v), 4)


rows = []
for tk, g in trades.groupby("ticker") if len(trades) else []:
    rows.append({
        "ticker": tk,
        "alpha_pos": _alpha_side(tk, "alpha_pos"), "alpha_neg": _alpha_side(tk, "alpha_neg"),
        "sigma": None if tk not in sigma_by_ticker else round(float(sigma_by_ticker[tk]), 4),
        "lambda": _lambda_of(tk),
        "ratings": by_ticker_ratings.get(tk, []),
        "trades": [{
            "cls": r["cls"], "sign": r["sign"], "date": date_index.get(int(r["trade_date"])),
            "entry_dev": round(float(r["entry_dev"]), 4), "pnl_pct": round(float(r["pnl"]), 4),
            "status": "win" if r["pnl"] > 0 else "loss",
            "signal_minute_idx": int(r["signal_minute"]), "entry_minute_idx": int(r["entry_minute"]),
            "exit_minute_idx": int(r["exit_minute"]),
        } for _, r in g.iterrows() if int(r["trade_date"]) in date_index],
    })
seen = {r["ticker"] for r in rows}
for tk in alpha.index:
    if tk in seen:
        continue
    ap, an = _alpha_side(tk, "alpha_pos"), _alpha_side(tk, "alpha_neg")
    if ap is None and an is None:
        continue
    rows.append({
        "ticker": tk, "alpha_pos": ap, "alpha_neg": an,
        "sigma": None if tk not in sigma_by_ticker else round(float(sigma_by_ticker[tk]), 4),
        "lambda": _lambda_of(tk),
        "ratings": by_ticker_ratings.get(tk, []), "trades": [],
    })

with gzip.open(ROLLING_PATH, "wt", encoding="utf-8", newline="\n") as f:
    json.dump({
        "meta": {
            "version": "opgcontinuum-rolling-perf-v1",
            "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
            "most_recent_session": recent_dates[-1] if recent_dates else None,
            "sessions_seen": len(recent_dates),
            "windows_days": WINDOWS_DAYS,
            "recent_dates": recent_dates,
            "position_usd": 1000,
            "min_dev": CFG["min_dev"],
            "pnl_basis": "raw Stack% move, entry = the 9:30 open gap (vs the PRIOR regular-session close) -> class exit (9:45/10:00/10:30, same calendar morning); rides the 9:20-9:25 gap - Long on a strong (pos) reading, Short on a weak (neg) one - the mirror of OPG-reversal's own fade thesis on the identical reading. Trades date by their own session (no midnight crossing in this family).",
            "lambda_basis": "per-ticker std(ddof=1) of the raw ENTRY-checkpoint (9:30 open print) Stack% "
                        "reading - the ticker's own historical open-gap dispersion. Gate-free, same "
                        "population as alpha.",
            "checkpoints": {k: [v.smin, v.tol] for k, v in CHECKPOINTS.items()},
            "signal_window": [SIGNAL_FROM_MIN, SIGNAL_TO_MIN],
        },
        "rows": rows,
    }, f, ensure_ascii=False)

print("written ->", OUT_DIR, "(", "opgcontinuum_ratings.csv", ROLLING_PATH.name, ")")
